# PHASE: W9-E

**SCIENTIFIC QUESTION:** What structure preserves source support and compactness?

**PREREQUISITES:** Complete FIT-frozen upstream phases; see runbook.

**INPUT ARTIFACTS:** configs/week9/w9e.yaml; exact paths in its data block and upstream frozen manifests.

**EXPECTED OUTPUTS:** phase JSONL cache, identity sidecar, report, summary CSV and run manifest. No synthetic fallbacks.

**CURRENT SPLIT:** FIT (default). INTERNAL-VAL is confirmation and has been inspected. DEV has been inspected. TEST has no notebook control.

**FULL RUN FLAG:** `RUN_FULL=False`. Full and smoke execution have separate explicit flags; Run All performs analysis only when real artifacts exist.


In [ ]:
from pathlib import Path
import sys, json, subprocess
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
while not (ROOT / 'src/week9').is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Open this notebook within CS787-RAG-Project')
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.week9.config import load_config, load_frozen
from src.week9.runner import rows_path, phase_dir, load_phase
RUN_FULL = False
RUN_SMOKE = False
FREEZE_FIT = False
SMOKE_LIMIT = 2
SPLIT = 'fit'
assert SPLIT in ('fit', 'internal_val', 'dev')
PHASE = 'e'
CONFIG = ROOT / 'configs/week9/w9e.yaml'
SCRIPT = ROOT / 'scripts/week9/run_w9e_extraction_faithfulness.py'
cfg = load_config(CONFIG)


In [ ]:
command = [str(ROOT / 'sara_env/bin/python'), str(SCRIPT), '--split', SPLIT,
           '--config', str(CONFIG), '--resume']
if RUN_FULL:
    if FREEZE_FIT:
        assert SPLIT == 'fit'
        command.append('--freeze')
    if PHASE == 'g':
        command.append('--collect-headroom-oof')
    subprocess.run(command, cwd=ROOT, check=True)
else:
    print('Full experiment disabled. Set RUN_FULL=True explicitly.')
if RUN_SMOKE:
    subprocess.run(command + ['--smoke-limit', str(SMOKE_LIMIT)], cwd=ROOT, check=True)


In [ ]:
directory = phase_dir(cfg, PHASE, SPLIT)
path = rows_path(cfg, PHASE, SPLIT)
if path.is_file():
    rows = load_phase(cfg, PHASE, SPLIT)
    run_manifest = json.loads((directory / f'w9{PHASE}_manifest.json').read_text())
    from src.week9.provenance import digest
    expected_config = dict(cfg, _seed=run_manifest['seed'])
    if digest(expected_config) != run_manifest['config_hash']:
        raise ValueError('Notebook configuration differs from the saved run')
    print('Complete:', run_manifest['complete'], 'Rows:', len(rows))
    print('Input/source fingerprints validated by load_phase.')
    frame = pd.DataFrame(rows)
    display(frame[[c for c in ['question_id','paper_id','policy','representation','retriever',
        'retrieval_width','evidence_k','f1','total_query_prompt_tokens'] if c in frame]].head(12))
    for table in sorted(directory.glob('*.csv')):
        print(table.name)
        display(pd.read_csv(table).head(30))
else:
    print('No full-run artifacts yet. Run the phase manually; no fallback metrics were created.')


In [ ]:
if path.is_file():
    import matplotlib.pyplot as plt
    if PHASE == 'a':
        table = pd.read_csv(directory / 'w9a_summary.csv')
        display(table.pivot_table(index=['representation','retriever'], columns='retrieval_width',
                                 values='all_gold_evidence_recall'))
        ax = table.pivot_table(index='representation', columns='retriever',
                              values='all_gold_evidence_recall').plot.bar(figsize=(12,5))
        ax.set_ylabel('All annotated evidence recall (paragraph proxy)')
    elif 'f1' in frame:
        group = 'evidence_k' if PHASE == 'b' else 'policy'
        ax = frame.groupby(group).f1.mean().plot.bar(figsize=(10,4))
        ax.set_ylabel('QASPER F1')
    else:
        ax = None
    if ax is not None:
        (directory / 'figures').mkdir(exist_ok=True)
        ax.figure.tight_layout()
        ax.figure.savefig(directory / 'figures' / f'w9{PHASE}_overview.png', dpi=160)
        plt.show()


## Manual decision checkpoint

Inspect completeness, source/config fingerprints, paired paper-bootstrap intervals and total query-time token costs before freezing. A smoke run cannot freeze a configuration. Summary tables come only from saved real rows. Exact-span support is separate from fact entailment, which is unmeasured unless explicitly judged. No next phase is launched here.

To regenerate summaries without loading the generator, run this phase script with `--analyze-only --resume`. Add `--freeze` only after a complete FIT run. H requires verified outer paper-fold outcomes and `--collect-oof`; see the runbook. Changes to source/config/model fingerprints require a new output root; incompatible caches are refused.
